In [6]:
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

df = pd.read_csv("tmdb_ds.csv")

df["title"] = df["title"].fillna("")

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["title"].apply(clean_text)

print(df[["title", "clean_text"]].head())

                                      title                     clean_text
0                                    Avatar                         avatar
1  Pirates of the Caribbean: At World's End  pirates caribbean world s end
2                                   Spectre                        spectre
3                     The Dark Knight Rises              dark knight rises
4                               John Carter                    john carter


In [7]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])

TF-IDF matrix shape: (4803, 5000)
Number of movies: 4803
Number of features: 5000


In [8]:
feature_names = tfidf.get_feature_names_out()

print("First 20 features:")
print(feature_names[:20])

if tfidf_matrix.shape[0] > 0:
    first_movie_vector = tfidf_matrix[0].toarray().flatten()

    nonzero_indices = first_movie_vector.nonzero()[0]

    sample_features = pd.DataFrame({
        "Word or Phrase": feature_names[nonzero_indices],
        "TF-IDF Score": first_movie_vector[nonzero_indices]
    })

    print(sample_features.sort_values(
        by="TF-IDF Score",
        ascending=False
    ).head(10))

First 20 features:
['000' '10' '10 days' '10 navarone' '10 things' '10 yuma' '100' '1000'
 '1000 corpses' '102' '102 dalmatians' '10th' '10th wolf' '11' '11 14'
 '12' '12 angry' '12 rounds' '13' '13th']
Empty DataFrame
Columns: [Word or Phrase, TF-IDF Score]
Index: []


TF-IDF converts cleaned movie descriptions into numerical vectors. The max_features parameter limits the vocabulary size, while ngram_range=(1, 2) allows the model to consider both individual words and two-word phrases. The resulting matrix contains one row per movie and one column per learned text feature. These vectors will be used to calculate movie similarity in the next task.